# 05. Agent Development Frameworks: Architecture vs Implementation

This course explores how modern agent frameworks encapsulate runtime mechanics. We compare four representative paradigms:
1. **Framework-Neutral Baseline (Raw Loop):** Application owns the loop, routing, state, and provider adapter.
2. **PydanticAI:** Framework owns the loop; specializes in type-safe outputs and dependency injection.
3. **LangGraph:** Framework models execution as a state machine graph; specializes in checkpointing and Human-in-the-Loop (HITL).
4. **OpenAI Agents SDK / Provider SDK:** Lightweight, provider-managed agent abstractions.

### Core Architectural Principles
- **Framework != Architecture:** The agent's capability boundary, state models, and safety invariants must be defined first.
- **The Grounding Invariant:** *No recommendation may rely on evidence the implementation did not retrieve.*
- **Persistence Semantics:** In-memory checkpointers (`MemorySaver`) demonstrate thread state resumption; durable persistence requires crash-safe backends (`SqliteSaver`, `PostgresSaver`).

In [ ]:
import json
import time
import os
import logging
from typing import List, Dict, Any, Optional, Set
from pydantic import BaseModel, Field, ConfigDict

# Centralized model configuration (Model/API capabilities evolve over time;
# official documentation at https://platform.openai.com/docs is the source of truth).
OPENAI_MODEL = 'gpt-4o-mini'
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger('Course05')
print(f'Environment initialized. Configured model: {OPENAI_MODEL}')

## Part 1 — Provider-Neutral Domain Models & Shared Tools

We define strict Pydantic inputs and outputs for our domain tools so that business logic remains completely decoupled from framework choices.

In [ ]:
# 1. Application Tool Schemas and Handlers
class HealthRequest(BaseModel):
    model_config = ConfigDict(extra='forbid')
    region: str = Field(..., description='Region identifier, e.g. EU, US')

class DeploymentRequest(BaseModel):
    model_config = ConfigDict(extra='forbid')
    service: str = Field(..., description='Microservice name, e.g. checkout')

def get_service_health(req: HealthRequest) -> str:
    logger.info(f'[TOOL] get_service_health(region={req.region})')
    return json.dumps({'evidence_id': f'health:{req.region}', 'status': 'DEGRADED', 'error_rate': '15%', 'symptom': 'Stripe timeout'})

def get_recent_deployments(req: DeploymentRequest) -> str:
    logger.info(f'[TOOL] get_recent_deployments(service={req.service})')
    return json.dumps({'evidence_id': f'deploy:{req.service}:dep_eu_114', 'latest_deployment_id': 'dep_eu_114', 'time': '20m ago', 'author': 'deploy-bot'})

# 2. Provider-Neutral Internal Runtime Types
class ToolCall(BaseModel):
    id: str
    name: str
    arguments_json: str

class ModelDecision(BaseModel):
    decision_summary: str = Field(..., description='Observable action rationale')
    tool_calls: List[ToolCall] = Field(default_factory=list)
    final_answer: Optional[str] = None

class AgentRunResult(BaseModel):
    final_answer: str
    evidence_retrieved: List[str]
    evidence_ids: List[str]
    steps: int
    tool_calls: int

print('Provider-neutral domain models and typed tools initialized.')

## Part 2 — The Framework-Neutral Baseline (Raw Loop)

To understand what frameworks automate, we build the loop manually.
### Grounded Multi-Step Trajectory
The agent executes a fully grounded trajectory:
1. `get_service_health(region='EU')` &rarr; observes degraded error rate
2. `get_recent_deployments(service='checkout')` &rarr; retrieves deployment `dep_eu_114`
3. Final recommendation synthesizing the evidence: `dep_eu_114` caused degradation; rollback recommended.

**Grounding Invariant:** The final recommendation only cites facts that were retrieved during the execution.

In [ ]:
class MockLLM:
    def __init__(self):
        self.turns = 0

    def chat(self, history: List[Dict[str, Any]]) -> ModelDecision:
        self.turns += 1
        if self.turns == 1:
            return ModelDecision(
                decision_summary='Investigate EU regional service health to confirm failure symptoms.',
                tool_calls=[ToolCall(id='tc_1', name='get_service_health', arguments_json='{"region": "EU"}')]
            )
        elif self.turns == 2:
            return ModelDecision(
                decision_summary='Service health is degraded. Retrieve recent checkout deployments to check for recent regressions.',
                tool_calls=[ToolCall(id='tc_2', name='get_recent_deployments', arguments_json='{"service": "checkout"}')]
            )
        else:
            return ModelDecision(
                decision_summary='Synthesize collected health and deployment evidence into final grounded recommendation.',
                final_answer='EU checkout service is DEGRADED (15% error rate) following deployment dep_eu_114 shipped 20m ago. Rollback of dep_eu_114 is recommended.'
            )

def raw_agent_loop(query: str, max_steps: int = 5) -> AgentRunResult:
    llm = MockLLM()
    history = [{'role': 'user', 'content': query}]
    evidence_retrieved = []
    evidence_ids = []
    steps = 0
    tool_calls_count = 0

    print(f'[Raw Loop] Starting investigation: "{query}"')
    while steps < max_steps:
        steps += 1
        decision: ModelDecision = llm.chat(history)
        history.append({'role': 'model', 'decision': decision})
        print(f'Step {steps} | Action rationale: {decision.decision_summary}')

        if decision.final_answer:
            print(f'[Terminal] Final Answer: {decision.final_answer}')
            return AgentRunResult(
                final_answer=decision.final_answer,
                evidence_retrieved=evidence_retrieved,
                evidence_ids=evidence_ids,
                steps=steps,
                tool_calls=tool_calls_count
            )

        for tc in decision.tool_calls:
            tool_calls_count += 1
            if tc.name == 'get_service_health':
                res = get_service_health(HealthRequest.model_validate_json(tc.arguments_json))
            elif tc.name == 'get_recent_deployments':
                res = get_recent_deployments(DeploymentRequest.model_validate_json(tc.arguments_json))
            else:
                res = json.dumps({'error': f'Unknown tool {tc.name}'})

            parsed_res = json.loads(res)
            if 'evidence_id' in parsed_res:
                evidence_ids.append(parsed_res['evidence_id'])
            evidence_retrieved.append(res)
            print(f'  [Observation] {res}')
            history.append({'role': 'tool', 'tool_id': tc.id, 'name': tc.name, 'content': res})

    raise TimeoutError('Step budget exceeded in raw loop.')

baseline_result = raw_agent_loop('Investigate EU checkout incident.')

# Verify Shared Grounding Invariant
def verify_grounding(result: AgentRunResult):
    """Enforce: No recommendation may rely on evidence the implementation did not retrieve."""
    if 'dep_eu_114' in result.final_answer:
        assert any('dep_eu_114' in ev for ev in result.evidence_retrieved), \
            'GROUNDING VIOLATION: Final answer cited deployment dep_eu_114, but deployment tool was never executed!'
    if 'DEGRADED' in result.final_answer or '15%' in result.final_answer:
        assert any('DEGRADED' in ev for ev in result.evidence_retrieved), \
            'GROUNDING VIOLATION: Final answer cited health degradation without checking health tool!'
    print('\nGrounding Invariant Verified: All cited facts match retrieved evidence.')

verify_grounding(baseline_result)

## Part 3 — PydanticAI: Typed Outputs & Dependency Injection

PydanticAI encapsulates the agent loop. Its defining architectural strengths are:
1. **Validated Typed Outputs (`output_type=FinalDecision`):** Guarantees structured return types without ad-hoc string parsing.
2. **Type-Safe Dependency Injection (`RunContext[AppDeps]`):** Injects application state (e.g. database connections, tenant credentials) safely into tool handlers.

In [ ]:
from dataclasses import dataclass

@dataclass
class AppDeps:
    environment: str
    api_token: str

class IncidentDiagnosis(BaseModel):
    requires_rollback: bool = Field(..., description='Whether a production deployment rollback is required')
    target_deployment: Optional[str] = Field(None, description='The deployment identifier to roll back')
    evidence_ids: List[str] = Field(default_factory=list, description='Explicit IDs of retrieved evidence justifying diagnosis')
    rationale_summary: str = Field(..., description='Observable diagnostic rationale')

try:
    from pydantic_ai import Agent, RunContext

    # PydanticAI Agent with typed output and dependency injection
    pydantic_agent = Agent(
        f'openai:{OPENAI_MODEL}',
        deps_type=AppDeps,
        output_type=IncidentDiagnosis,
        system_prompt='You are an incident diagnostic assistant. Gather evidence with tools before recommending actions.'
    )

    @pydantic_agent.tool
    def check_health(ctx: RunContext[AppDeps], region: str) -> str:
        logger.info(f'[PydanticAI Tool] Environment={ctx.deps.environment} checking region={region}')
        return get_service_health(HealthRequest(region=region))

    @pydantic_agent.tool
    def check_deployments(ctx: RunContext[AppDeps], service: str) -> str:
        logger.info(f'[PydanticAI Tool] Environment={ctx.deps.environment} checking service={service}')
        return get_recent_deployments(DeploymentRequest(service=service))

    print('PydanticAI agent initialized successfully with output_type=IncidentDiagnosis.')
    
    if os.getenv('OPENAI_API_KEY'):
        deps = AppDeps(environment='production', api_token='sec_token_99')
        run_res = pydantic_agent.run_sync('Investigate EU checkout failures.', deps=deps)
        # Access the validated structured data
        print('\nLive PydanticAI Output:', run_res.data)
except ImportError:
    print('pydantic-ai package is not installed in the current environment. Architecture pattern defined above.')

## Part 4 — LangGraph: State Machine Graph & Checkpoint Semantics

LangGraph models the agent loop as an explicit directed graph with state checkpointing.

> [!IMPORTANT]
> **Persistence Clarification:** `MemorySaver` demonstrates in-memory checkpoint/resume semantics within a single process. > True crash-safe durability across process restarts requires a persistent checkpointer backend (such as `SqliteSaver` or `PostgresSaver`).

In [ ]:
try:
    from typing import Annotated
    from typing_extensions import TypedDict
    from langgraph.graph import StateGraph, START, END
    from langgraph.graph.message import add_messages
    from langgraph.checkpoint.memory import MemorySaver

    class GraphState(TypedDict):
        messages: Annotated[list, add_messages]
        human_approved: bool
        evidence_collected: list

    def agent_decide_node(state: GraphState):
        logger.info('[LangGraph Node: Agent] Evaluating gathered evidence.')
        return {
            'messages': [{'role': 'assistant', 'content': 'Diagnosis: Deployment dep_eu_114 caused degradation. Proposing rollback.'}],
            'evidence_collected': ['health:EU', 'deploy:checkout:dep_eu_114']
        }

    def human_review_node(state: GraphState):
        logger.info('[LangGraph Node: Review] Human on-call approved rollback.')
        return {'human_approved': True}

    builder = StateGraph(GraphState)
    builder.add_node('agent', agent_decide_node)
    builder.add_node('review', human_review_node)
    builder.add_edge(START, 'agent')
    builder.add_edge('agent', 'review')
    builder.add_edge('review', END)

    # In-memory checkpointer demonstrates checkpointing & interrupt/resume mechanics
    checkpointer = MemorySaver()
    graph = builder.compile(checkpointer=checkpointer, interrupt_before=['review'])
    
    config = {'configurable': {'thread_id': 'incident_thread_101'}}
    
    print('\n--- 1. Graph Execution Pausing at Human Review Breakpoint ---')
    for event in graph.stream({'messages': [{'role': 'user', 'content': 'Investigate checkout'}], 'human_approved': False, 'evidence_collected': []}, config):
        pass
    
    checkpoint_state = graph.get_state(config)
    print('Thread is paused before node:', checkpoint_state.next)
    print('Current checkpoint values:', checkpoint_state.values)
    
    print('\n--- 2. Resuming Paused Thread After Human Approval ---')
    # Resumes execution from the exact checkpoint without re-running prior nodes
    for event in graph.stream(None, config):
        pass
    
    final_state = graph.get_state(config)
    print('Execution complete. Human approved status:', final_state.values['human_approved'])
    assert final_state.values['human_approved'] == True
except ImportError:
    print('langgraph is not installed. Graph architecture pattern defined above.')

## Part 5 — OpenAI Responses API & Agents SDK Patterns

For lightweight setups, developers often interact with provider SDKs directly or through minimal agent abstractions.
- **Raw Responses API:** Current `client.responses.create` and `client.responses.parse` multi-step tool loop.
- **OpenAI Agents SDK (`agents`):** Lightweight `Agent` + `Runner.run_sync` abstraction.

In [ ]:
# 1. OpenAI Raw Responses API Pattern
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print('No OPENAI_API_KEY detected. Skipping live OpenAI calls.')
else:
    from openai import OpenAI
    client = OpenAI(api_key=api_key)

    openai_tools = [
        {'type': 'function', 'name': 'get_service_health', 'description': 'Check region health', 'parameters': HealthRequest.model_json_schema()},
        {'type': 'function', 'name': 'get_recent_deployments', 'description': 'Check recent deployments', 'parameters': DeploymentRequest.model_json_schema()}
    ]

    conversation_input = [{'role': 'user', 'content': 'Investigate EU checkout failures.'}]

    print(f'\n--- Running Live OpenAI Responses API Loop ({OPENAI_MODEL}) ---')
    for step in range(1, 4):
        resp = client.responses.create(
            model=OPENAI_MODEL,
            instructions='You are an incident response assistant. Check service health and deployments before recommending actions.',
            input=conversation_input,
            tools=openai_tools
        )

        fn_calls = [item for item in resp.output if getattr(item, 'type', None) == 'function_call' or hasattr(item, 'call_id')]
        if fn_calls:
            tc = fn_calls[0]
            try:
                args = json.loads(tc.arguments) if isinstance(tc.arguments, str) else tc.arguments
            except Exception:
                args = {}
            print(f'Step {step} | Model requested tool: {tc.name}({args})')
            if tc.name == 'get_service_health':
                obs = get_service_health(HealthRequest.model_validate(args))
            elif tc.name == 'get_recent_deployments':
                obs = get_recent_deployments(DeploymentRequest.model_validate(args))
            else:
                obs = json.dumps({'error': 'Unknown tool'})
            
            print(f'  Observation: {obs}')
            call_id = getattr(tc, 'call_id', f'call_{step}')
            conversation_input.append({'type': 'function_call', 'call_id': call_id, 'name': tc.name, 'arguments': json.dumps(args)})
            conversation_input.append({'type': 'function_call_output', 'call_id': call_id, 'output': obs})
        else:
            print('\nFinal Model Answer:', resp.output_text)
            break

    # 2. Structured Output via Responses API
    print(f'\n--- Live Structured Output Parsing ({OPENAI_MODEL}) ---')
    try:
        structured_resp = client.responses.parse(
            model=OPENAI_MODEL,
            input=conversation_input,
            text_format=IncidentDiagnosis
        )
        parsed_output = structured_resp.output_parsed
        print('Parsed IncidentDiagnosis (Responses API):', parsed_output)
    except Exception:
        structured_resp = client.beta.chat.completions.parse(
            model=OPENAI_MODEL,
            messages=[{'role': 'user', 'content': 'Investigate EU checkout failures.'}],
            response_format=IncidentDiagnosis
        )
        parsed_output: IncidentDiagnosis = structured_resp.choices[0].message.parsed
        print('Parsed IncidentDiagnosis (Chat fallback):', parsed_output)

# 3. OpenAI Agents SDK Pattern
try:
    from agents import Agent, Runner
    print('\n--- OpenAI Agents SDK Pattern ---')
    agent_instance = Agent(
        name='IncidentDiagnosticAgent',
        instructions='Diagnose incidents by gathering health and deployment evidence.',
        model=OPENAI_MODEL
    )
    print('OpenAI Agents SDK Agent initialized successfully:', agent_instance.name)
except ImportError:
    print('\nNote: openai-agents package not installed in the local environment. Architecture pattern defined above.')

## Part 6 — Objective Framework Comparison Matrix

We evaluate the four paradigms across critical production dimensions.

In [ ]:
import pandas as pd

framework_matrix = [
    {
        'Framework': 'Raw / Provider-Neutral',
        'Loop Ownership': 'Application Code',
        'Output Typing': 'Manual Pydantic Validation',
        'State Visibility': 'Full (Direct Python State)',
        'Persistence': 'Custom (Application DB)',
        'HITL': 'Native (Application Control Flow)',
        'Portability': 'High (Universal)',
        'Complexity': 'Low'
    },
    {
        'Framework': 'PydanticAI',
        'Loop Ownership': 'Framework Runtime',
        'Output Typing': 'Strict (output_type & Deps)',
        'State Visibility': 'Moderate (RunContext / Messages)',
        'Persistence': 'Optional (Message History)',
        'HITL': 'Manual / Message Resumption',
        'Portability': 'High (Multi-Model Support)',
        'Complexity': 'Medium'
    },
    {
        'Framework': 'LangGraph',
        'Loop Ownership': 'State Machine Graph',
        'Output Typing': 'Manual Node Schemas',
        'State Visibility': 'High (Graph State & Channels)',
        'Persistence': 'First-Class (Checkpointers)',
        'HITL': 'Built-in (interrupt_before/after)',
        'Portability': 'High (LangChain Ecosystem)',
        'Complexity': 'High'
    },
    {
        'Framework': 'OpenAI Agents SDK / Raw SDK',
        'Loop Ownership': 'SDK Runner / App Loop',
        'Output Typing': 'Built-in (.parse() / Pydantic)',
        'State Visibility': 'Moderate (Messages / Run Items)',
        'Persistence': 'External / Custom',
        'HITL': 'Manual Hand-offs',
        'Portability': 'Low-to-Medium (OpenAI Focused)',
        'Complexity': 'Low'
    }
]

df_comparison = pd.DataFrame(framework_matrix)
print('=== AGENT DEVELOPMENT FRAMEWORKS COMPARISON MATRIX ===')
print(df_comparison.to_string(index=False))